In [4]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset --------
class ModeAndFeatureDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None, excluded_ids_path=None, max_items=None):
        self.items = []
        self.distances = {}
        self.excluded_ids = set()
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)
        if excluded_ids_path:
            with open(excluded_ids_path, encoding='utf-8') as f:
                self.excluded_ids = set([line.strip() for line in f if line.strip().isdigit()])

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if not sid.isdigit() or sid in self.excluded_ids:
                continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) == 0: continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann['sequence']
            min_len = min(len(files), len(seq))
            if min_len < 15: continue

            own_speeds = np.array([f['OwnSpeed'] / 3.6 for f in seq], dtype=np.float32)
            tgt_speeds = np.array([f['TgtSpeed_ref'] / 3.6 for f in seq], dtype=np.float32)
            angles = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            dists_all = [self.distances.get(sid, {}).get(str(i), 0.0) for i in range(min_len)]
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                if max_items and len(self.items) >= max_items:
                    return

                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]
                t = tgt_speeds[i:i+15]

                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                rel_acc = d2 - np.mean(d2)

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                rel_speed = np.mean(t - s)
                feature = np.concatenate([modes, d, s, a, s1, d1, d2, rel_acc] + d_smooths)
                self.items.append((feature.astype(np.float32), rel_speed, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feature, tgt, sid = self.items[idx]
        return torch.tensor(feature), torch.tensor(tgt, dtype=torch.float32), sid

# -------- Collate --------
def collate_fn(batch):
    feats, tgts, sids = zip(*batch)
    return torch.stack(feats), torch.tensor(tgts), list(sids)

# -------- LSTM Model --------
class ExtendedLSTMModel(nn.Module):
    def __init__(self, in_dim=225, seq_len=15, hidden_size=64, num_layers=1):
        super().__init__()
        self.seq_len = seq_len
        self.feature_dim = in_dim // seq_len
        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B = x.size(0)
        x = x.view(B, self.seq_len, self.feature_dim)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.fc(last).squeeze(1)

# -------- Training Loop --------
def train_extended_model(dataset, save_path="420_2.pth"):
    scenes = sorted(list(set([item[-1] for item in dataset.items])))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ExtendedLSTMModel(in_dim=train_ds[0][0].shape[0]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-2)
    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 150
    patience_counter = 0

    for epoch in range(250):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"\u2705 Model saved to {save_path} (val_loss={val_loss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\u23F9 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- Run Training --------
if __name__ == "__main__":
    crop_root = "../train_retry/n/train_crops"
    annot_root = "../train/train_annotations"
    distance_json_path = "../train_retry/filtered_distance_estimates.json"
    excluded_ids_path = "../train_retry/excluded_scene_ids.txt"

    dataset = ModeAndFeatureDataset(
        crop_root=crop_root,
        annot_root=annot_root,
        distance_json_path=distance_json_path,
        excluded_ids_path=excluded_ids_path,
        max_items=10000
    )

    model = train_extended_model(dataset, save_path="420_2.pth")


[Train 1]: 100%|██████████| 62/62 [00:00<00:00, 220.28it/s]


Epoch 1 | Train Loss: 0.6567 | Val Loss: 0.6196
✅ Model saved to 420_2.pth (val_loss=0.6196)


[Train 2]: 100%|██████████| 62/62 [00:00<00:00, 222.66it/s]


Epoch 2 | Train Loss: 0.6551 | Val Loss: 0.6208


[Train 3]: 100%|██████████| 62/62 [00:00<00:00, 227.96it/s]


...

Epoch 149 | Train Loss: 0.6548 | Val Loss: 0.6230


[Train 150]: 100%|██████████| 62/62 [00:00<00:00, 222.37it/s]


Epoch 150 | Train Loss: 0.6547 | Val Loss: 0.6230


[Train 151]: 100%|██████████| 62/62 [00:00<00:00, 221.23it/s]


Epoch 151 | Train Loss: 0.6547 | Val Loss: 0.6230
⏹ Early stopping at epoch 151
